# Barter-RS: Trading Statistics & Performance Analysis

This notebook demonstrates how to use barter's built-in statistical engine to
compute trading performance metrics from synthetic position and balance data.

## Topics Covered
- Setting up `EngineState` with initial balances
- Using `TradingSummaryGenerator` to track PnL
- Feeding balance updates and closed positions
- Generating annualised performance summaries (Sharpe, Sortino, Calmar, Drawdown)

In [ ]:
:dep barter = { version = "0.12" }
:dep barter-execution = { version = "0.7" }
:dep barter-instrument = { version = "0.3" }
:dep barter-integration = { version = "0.10" }
:dep chrono = { version = "0.4", features = ["serde"] }
:dep rust_decimal = { version = "1.36", features = ["maths"] }
:dep rust_decimal_macros = "1.29"
:dep smol_str = "0.3"

## 1. Define Instruments and Initial State

`IndexedInstruments` is the central registry mapping exchanges, assets, and
instruments to compact integer indices for O(1) lookups.

In [ ]:
use barter::{
    engine::state::{
        EngineState, global::DefaultGlobalData,
        instrument::data::DefaultInstrumentMarketData,
        position::PositionExited, trading::TradingState,
    },
    statistic::{summary::TradingSummaryGenerator, time::Annual365},
};
use barter_execution::{
    balance::{AssetBalance, Balance},
    trade::{AssetFees, TradeId},
};
use barter_instrument::{
    Side, Underlying,
    asset::{AssetIndex, QuoteAsset},
    exchange::ExchangeId,
    index::IndexedInstruments,
    instrument::{Instrument, InstrumentIndex},
};
use barter_integration::collection::snapshot::Snapshot;
use chrono::{DateTime, Days, Utc};
use rust_decimal::Decimal;
use rust_decimal_macros::dec;
use smol_str::SmolStr;

// Risk-free rate for Sharpe/Sortino calculations
const RISK_FREE_RETURN: Decimal = dec!(0.05);

// Build instrument index: BTC/USDT and ETH/USDT on Binance Spot
let instruments = IndexedInstruments::builder()
    .add_instrument(Instrument::spot(
        ExchangeId::BinanceSpot,
        "binance_spot_btc_usdt",
        "BTCUSDT",
        Underlying::new("btc", "usdt"),
        None,
    ))
    .add_instrument(Instrument::spot(
        ExchangeId::BinanceSpot,
        "binance_spot_eth_usdt",
        "ETHUSDT",
        Underlying::new("eth", "usdt"),
        None,
    ))
    .build();

println!("Indexed {} instruments", instruments.instruments().len());
println!("Indexed {} assets", instruments.assets().len());

## 2. Build Engine State with Balances

The `EngineState` builder initialises the state with starting balances per
exchange and asset. This is the same state structure used by the live Engine.

In [ ]:
let time_start = Utc::now();

let state = EngineState::builder(&instruments, DefaultGlobalData::default(), |_| {
    DefaultInstrumentMarketData::default()
})
.time_engine_start(time_start)
.trading_state(TradingState::Enabled)
.balances([
    (ExchangeId::BinanceSpot, "usdt", Balance::new(dec!(10_000.0), dec!(10_000.0))),
    (ExchangeId::BinanceSpot, "btc",  Balance::new(dec!(0.1), dec!(0.1))),
    (ExchangeId::BinanceSpot, "eth",  Balance::new(dec!(1.0), dec!(1.0))),
])
.build();

println!("EngineState built with starting balances.");

## 3. Initialise TradingSummaryGenerator

The generator tracks balance changes and closed positions over time, then
computes portfolio-level and per-instrument statistics.

In [ ]:
let mut summary_gen = TradingSummaryGenerator::init(
    RISK_FREE_RETURN,
    time_start,
    time_start,
    &state.instruments,
    &state.assets,
);

println!("TradingSummaryGenerator initialised.");
println!("Risk-free rate: {RISK_FREE_RETURN}");

## 4. Feed Synthetic Trading Events

We simulate a series of trades:
1. Buy BTC (spend 1000 USDT) → sell for profit (+2000)
2. Buy BTC again → sell for profit (+1000)
3. Buy BTC → sell at a loss (-2000)
4. Buy ETH → sell at a loss (-1000)
5. Buy ETH → sell for small profit (+500)

In [ ]:
// Helper to create a balance update
fn balance_update(usdt_total: Decimal, day_offset: u64, base_time: DateTime<Utc>) -> Snapshot<AssetBalance<AssetIndex>> {
    Snapshot::new(AssetBalance {
        asset: AssetIndex(2), // usdt
        balance: Balance::new(usdt_total, usdt_total),
        time_exchange: base_time.checked_add_days(Days::new(day_offset)).unwrap(),
    })
}

// Helper to create a closed position
fn closed_position(
    instrument: InstrumentIndex, pnl: Decimal,
    qty: Decimal, day_enter: u64, day_exit: u64,
    trade_ids: Vec<&str>, base_time: DateTime<Utc>,
) -> PositionExited<QuoteAsset, InstrumentIndex> {
    PositionExited {
        instrument,
        side: Side::Buy,
        price_entry_average: dec!(1.0),
        quantity_abs_max: qty,
        pnl_realised: pnl,
        fees_enter: AssetFees::default(),
        fees_exit: AssetFees::default(),
        time_enter: base_time.checked_add_days(Days::new(day_enter)).unwrap(),
        time_exit: base_time.checked_add_days(Days::new(day_exit)).unwrap(),
        trades: trade_ids.into_iter().map(|id| TradeId(SmolStr::new(id))).collect(),
    }
}

let btc = InstrumentIndex(0);
let eth = InstrumentIndex(1);

// Trade 1: BTC buy (10000 -> 9000 USDT), then sell for profit (9000 -> 12000)
summary_gen.update_from_balance(balance_update(dec!(9000), 1, time_start).as_ref());
summary_gen.update_from_balance(balance_update(dec!(12000), 2, time_start).as_ref());
summary_gen.update_from_position(&closed_position(btc, dec!(2000), dec!(1000), 1, 2, vec!["1","2"], time_start));
println!("Trade 1 (BTC): +2000 USDT profit");

// Trade 2: BTC buy (12000 -> 10000), sell (10000 -> 13000)
summary_gen.update_from_balance(balance_update(dec!(10000), 2, time_start).as_ref());
summary_gen.update_from_balance(balance_update(dec!(13000), 3, time_start).as_ref());
summary_gen.update_from_position(&closed_position(btc, dec!(1000), dec!(2000), 2, 3, vec!["3","4"], time_start));
println!("Trade 2 (BTC): +1000 USDT profit");

// Trade 3: BTC loss (13000 -> 8000 -> 11000)
summary_gen.update_from_balance(balance_update(dec!(8000), 4, time_start).as_ref());
summary_gen.update_from_balance(balance_update(dec!(11000), 5, time_start).as_ref());
summary_gen.update_from_position(&closed_position(btc, dec!(-2000), dec!(2000), 4, 5, vec!["5","6"], time_start));
println!("Trade 3 (BTC): -2000 USDT loss");

// Trade 4: ETH loss (11000 -> 6000 -> 5000 -> 10000)
summary_gen.update_from_balance(balance_update(dec!(6000), 6, time_start).as_ref());
summary_gen.update_from_balance(balance_update(dec!(5000), 7, time_start).as_ref());
summary_gen.update_from_balance(balance_update(dec!(10000), 8, time_start).as_ref());
summary_gen.update_from_position(&closed_position(eth, dec!(-1000), dec!(6000), 6, 8, vec!["7","8","9"], time_start));
println!("Trade 4 (ETH): -1000 USDT loss");

// Trade 5: ETH small profit (10000 -> 7000 -> 10500)
summary_gen.update_from_balance(balance_update(dec!(7000), 10, time_start).as_ref());
summary_gen.update_from_balance(balance_update(dec!(10500), 11, time_start).as_ref());
summary_gen.update_from_position(&closed_position(eth, dec!(500), dec!(6000), 10, 11, vec!["10","11"], time_start));
println!("Trade 5 (ETH): +500 USDT profit");

println!("\nAll synthetic events fed.");

## 5. Generate & Display Trading Summary

Call `.generate(Annual365)` for crypto-style 24/7 annualisation, or
`.generate(Annual252)` for traditional equity-style (252 trading days).

In [ ]:
let summary = summary_gen.generate(Annual365);

// Print the full summary table to the terminal
summary.print_summary();

## Available Metrics

The `TradingSummary` includes per-instrument and portfolio-wide statistics:

| Metric | Description |
|--------|-------------|
| **Total PnL** | Net realised profit/loss |
| **Win Rate** | Percentage of profitable trades |
| **Sharpe Ratio** | Risk-adjusted return (vs risk-free rate) |
| **Sortino Ratio** | Downside-risk-adjusted return |
| **Calmar Ratio** | Return / max drawdown |
| **Max Drawdown** | Largest peak-to-trough decline |
| **Rate of Return** | Annualised return percentage |
| **Profit Factor** | Gross profit / gross loss |

### Time Intervals

- `Annual365` — 24/7 crypto markets (365 days/year)
- `Annual252` — traditional equity markets (252 trading days/year)
- `Daily` — daily statistics without annualisation